In [0]:
%sql
USE CATALOG ws_banking_etl2;
CREATE SCHEMA IF NOT EXISTS silver;

In [0]:
from pyspark.sql.functions import col, regexp_replace, to_timestamp

transactions = spark.read.table("bronze.transactions_data")
fraud = spark.read.table("bronze.train_fraud_labels")
mcc = spark.read.table("bronze.mcc_codes")

# --- Clean transactions ---
transactions_clean = (
    transactions.withColumn("transaction_id", col("id").cast("int"))
        .withColumn("amount", col("amount").cast("decimal(19,4)"))
        .withColumn("transaction_date", to_timestamp(col("date")))
        .drop("id", "date")
)

# --- Clean fraud ---
from pyspark.sql.functions import when, lower

fraud_clean = (
    fraud
    .withColumn("transaction_id", col("id").cast("int"))
    .withColumn(
        "is_fraud",
        when(lower(col("target")) == "yes", True)
        .when(lower(col("target")) == "no", False)
        .otherwise(None)
    )
    .select("transaction_id", "is_fraud")
)

# --- Clean MCC ---
mcc_clean = (
    mcc
    .withColumn("mcc", col("mcc_code").cast("short"))
    .select("mcc", col("description").alias("mcc_description"))
)

# --- Join for enrichment---
transactions_silver = (
    transactions_clean
    .join(fraud_clean, on="transaction_id", how="left")
    .join(mcc_clean, on="mcc", how="left")
)

# --- Handle nulls ---
transactions_silver = transactions_silver.fillna({"is_fraud": False})

# --- Remove duplicates ---
transactions_silver = transactions_silver.dropDuplicates()

# --- Write ---
transactions_silver.write.format("delta").mode("overwrite").saveAsTable("silver.transactions")

transactions_silver.show(5)
print(f"silver transactions row count:{transactions_silver.count()}")

+----+--------------+---------+-------+--------+-----------------+-----------+-------------+--------------+-------+------+-------------------+--------+--------------------+
| mcc|transaction_id|client_id|card_id|  amount|         use_chip|merchant_id|merchant_city|merchant_state|    zip|errors|   transaction_date|is_fraud|     mcc_description|
+----+--------------+---------+-------+--------+-----------------+-----------+-------------+--------------+-------+------+-------------------+--------+--------------------+
|5912|      12887584|      524|   2263|105.4200|Swipe Transaction|      81833| Saint Albans|            WV|25177.0|  NULL|2013-06-14 15:33:00|   false|Drug Stores and P...|
|5411|      12888065|     1313|   4183| 73.7100|Swipe Transaction|      61746|  Sutersville|            PA|15083.0|  NULL|2013-06-14 17:34:00|   false|Grocery Stores, S...|
|5411|      12888632|     1723|   4099| 38.7300|Swipe Transaction|      50783|   White Hall|            IL|62092.0|  NULL|2013-06-14 21

In [0]:
from pyspark.sql.functions import regexp_replace

users = spark.read.table("bronze.users_data")

users_silver = (
    users
    .withColumnRenamed("id", "user_id")
    .withColumn("per_capita_income", regexp_replace(col("per_capita_income"), r"\$|,", "").cast("decimal(12,2)"))
    .withColumn("yearly_income", regexp_replace(col("yearly_income"), r"\$|,", "").cast("decimal(12,2)"))
    .withColumn("total_debt", regexp_replace(col("total_debt"), r"\$|,", "").cast("decimal(12,2)"))
    .dropDuplicates()
)

users_silver.write.format("delta").mode("overwrite").saveAsTable("silver.users")

users_silver.show(5)
print(f"silver users row count:{users_silver.count()}")

+-------+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|user_id|current_age|retirement_age|birth_year|birth_month|gender|             address|latitude|longitude|per_capita_income|yearly_income|total_debt|credit_score|num_credit_cards|
+-------+-----------+--------------+----------+-----------+------+--------------------+--------+---------+-----------------+-------------+----------+------------+----------------+
|    825|         53|            66|      1966|         11|Female|       462 Rose Lane|   34.15|  -117.76|         29278.00|     59696.00| 127613.00|         787|               5|
|   1746|         53|            68|      1966|         12|Female|3606 Federal Boul...|   40.76|   -73.74|         37891.00|     77254.00| 191349.00|         701|               5|
|   1718|         81|            67|      1938|         11|Female|     766 Third Drive|   34.02|  -1

In [0]:
from pyspark.sql.functions import to_date, lower

cards = spark.read.table("bronze.cards_data")

cards_silver = (
    cards
    .withColumnRenamed("id", "card_id")
    .withColumn("acct_open_date", to_date(col("acct_open_date"), "MM/yyyy"))
    .withColumn("has_chip",
    when(lower(col("has_chip")) == "yes", True)
    .when(lower(col("has_chip")) == "no", False)
    .otherwise(None)
)
    .dropDuplicates()
)

cards_silver.write.format("delta").mode("overwrite").saveAsTable("silver.cards")
print(f"silver cards row count:{cards_silver.count()}")


silver cards row count:6146


In [0]:
mcc = spark.read.table("bronze.mcc_codes")

mcc_silver = (
    mcc
    .withColumn("mcc", col("mcc_code").cast("short"))
    .select("mcc", col("description").alias("mcc_description"))
    .dropDuplicates()
)

mcc_silver.write.format("delta").mode("overwrite").saveAsTable("silver.mcc")
print(f"silver mcc row count:{mcc_silver.count()}")

silver mcc row count:109


In [0]:
fraud = spark.read.table("bronze.train_fraud_labels")

fraud_silver = (
    fraud
    .withColumn("transaction_id", col("id").cast("int"))
    .withColumn(
        "is_fraud",
        when(lower(col("target")) == "yes", True)
        .when(lower(col("target")) == "no", False)
    )
    .select("transaction_id", "is_fraud")
    .dropDuplicates()
)

fraud_silver.write.format("delta").mode("overwrite").saveAsTable("silver.fraud_labels")
print(f"silver fraud row count:{fraud_silver.count()}")

silver fraud row count:8914963


In [0]:
# Null check
from pyspark.sql.functions import col

# critical columns
null_txn = transactions_silver.filter(
    col("transaction_id").isNull() |
    col("amount").isNull()
).count()

print(f"Null critical rows: {null_txn}")

Null critical rows: 0


In [0]:
# Duplicate checks
duplicates = (
    transactions_silver
    .groupBy("transaction_id")
    .count()
    .filter("count > 1")
    .count()
)

print(f"Duplicate transaction_ids: {duplicates}")

Duplicate transaction_ids: 0
